<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# Instructor Demo Setup

This notebook materializes the large reference tables used by the Module 2 performance demo (`2.2 Demo - Interoperability and Performance Benefits of Managed Tables`). It builds:

- `store_sales_unclustered` - a deliberately shuffled copy of `samples.tpcds_sf1000.store_sales` (~2.8 billion rows). Used as the baseline for the Liquid Clustering performance comparison.
- `store_sales_clustered` - the same data laid out by Liquid Clustering on `(ss_sold_date_sk, ss_item_sk)`. Used as the optimized comparison for the perf demo.

Both tables persist across course sessions; this notebook is intended to be run once per instructor environment ahead of class.

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Run Before Class - Up to 10 Minutes</strong>
            <p style="margin: 8px 0 0 0; color: #333; font-size: 1em;">This notebook materializes ~2.8 billion rows twice. Each CTAS takes <b>several minutes</b> on a Serverless SQL Warehouse - allow up to <b>10 minutes total</b>. Run this notebook <b>before class starts</b> so the perf demo runs cleanly during the session.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Run AFTER <code>0 - Required Setup</code></strong>
            <p style="margin: 8px 0 0 0; color: #333;"><code>0 - Required Setup</code> creates the catalog, the <code>data_interoperability_tpcds</code> schema, and the dimension views. This notebook depends on that schema already existing - run <code>0 - Required Setup</code> first.</p>
        </div>
    </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li><strong>Serverless Compute, Version 5</strong>: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2272B4">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #37474f; font-size: 1.1em;">Step 1: Use the Course Schema</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Run <code>Classroom-Setup-Common</code> to set the active catalog, then <code>USE SCHEMA</code> to enter the schema created by <code>0 - Required Setup</code>.</p>
        </div>
    </div>
</div>

In [0]:
%run ./Includes/Classroom-Setup-Common

In [0]:
USE SCHEMA data_interoperability_tpcds;

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #37474f; font-size: 1.1em;">Step 2: Build the Unclustered Reference Table</strong>
            <p style="margin: 8px 0 0 0; color: #333;">CTAS <code>samples.tpcds_sf1000.store_sales</code> into <code>store_sales_unclustered</code> with <code>ORDER BY rand()</code> to deliberately shuffle the data. The source is partitioned by date, so without the shuffle a date-range filter would prune almost everything before clustering even mattered. Shuffling produces a true unclustered baseline.</p>
        </div>
    </div>
</div>

In [0]:
-- Unclustered reference copy of TPC-DS store_sales (~2.8B rows)
-- This is the baseline for Module 2's clustering performance demo
CREATE OR REPLACE TABLE store_sales_unclustered AS
SELECT * FROM samples.tpcds_sf1000.store_sales
ORDER BY rand();

SELECT 'store_sales_unclustered' AS table_name, COUNT(*) AS row_count
FROM store_sales_unclustered;

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #37474f; font-size: 1.1em;">Step 3: Build the Clustered Comparison Table</strong>
            <p style="margin: 8px 0 0 0; color: #333;">CTAS the same data with <b>Liquid Clustering</b> on <code>(ss_sold_date_sk, ss_item_sk)</code> - the columns the perf demo filters on. This is what gives the demo a meaningful pruning advantage over the unclustered baseline.</p>
        </div>
    </div>
</div>

In [0]:
-- Liquid Clustering on the columns we plan to filter on
CREATE OR REPLACE TABLE store_sales_clustered
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
AS SELECT * FROM store_sales_unclustered;

SELECT 'store_sales_clustered' AS table_name, COUNT(*) AS row_count FROM store_sales_clustered;

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">Setup Complete</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Both reference tables are materialized. The <b>2.2 Demo</b> can now run cleanly without students waiting on multi-billion-row CTASes during class.</p>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>